In [1]:
# Milestone 3 · Step 1 — project code from GitHub + the OpenAI key from Kaggle Secrets
!git clone -q https://github.com/jamalla/arabic-book-recommender-agent.git repo
!pip install -q "openai>=1.66"
import os, sys
from kaggle_secrets import UserSecretsClient

os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY")  # never printed
sys.path.insert(0, "repo")

from bookrec.agent import BookAgent, SYSTEM_PROMPT_V1
from bookrec.search import BookSearch

agent = BookAgent(search=BookSearch("repo/data/catalog.csv"))
print("model:", agent.model)
print("prompt v1:\n" + SYSTEM_PROMPT_V1)

model: gpt-6-luna
prompt v1:
You are a friendly bookstore assistant for Arabic readers.
Use the search_books tool to look up books in our catalog, then recommend books that match
the reader's request, with one short sentence on why each one fits.
Always answer in Arabic.


In [2]:
# Step 2 — ask the agent, and look INSIDE the loop: which searches did it run, what came back?
def show(question):
    result = agent.answer(question)
    print("❓", question)
    for i, step in enumerate(result["steps"], 1):
        print(f"\n🔎 tool call {i}: search_books({step['query']!r})")
        for book in step["books"]:
            print(f"   {book['score']:.2f}  {book['book_id']}  {book['title']} — {book['author']}")
    print("\n💬 answer:\n" + result["answer"])
    return result

r1 = show("أريد رواية تاريخية")

❓ أريد رواية تاريخية

🔎 tool call 1: search_books('رواية تاريخية')
   0.18  B192  تاريخ العرب القديم — توفيق برو
   0.16  B012  تجليات جبل — قادة جليد
   0.15  B168  المسرح في لبنان الشمالي منذ نشأته حتى نهاية القرن العشرين — نزيه كبارة
   0.13  B187  التاريخ الخفي — باسم رعد
   0.13  B015  عزلة الأحبة — باولو جيوردانو

💬 answer:
أقرب خيار في الفهرس هو **«تجليات جبل» لقادة جليد**؛ رواية تستلهم التراث الجزائري وعاداته وتقاليده، وإن لم تُوصَف صراحةً بأنها رواية تاريخية.


In [3]:
# Step 3 — a normal request and two TRAPS: a book we don't have, and a famous author the model knows
r2 = show("أبحث عن كتاب يساعدني في تربية أطفالي")
print("\n" + "=" * 100 + "\n")
r3 = show("هل عندكم رواية هاري بوتر؟")
print("\n" + "=" * 100 + "\n")
r4 = show("رشح لي روايات لنجيب محفوظ")

❓ أبحث عن كتاب يساعدني في تربية أطفالي

🔎 tool call 1: search_books('كتاب عن تربية الأطفال والأسرة، نصائح عملية للوالدين')
   0.27  B135  صورة الطفولة في التربية الإسلامية — إبراهيم الخطيب ، أحمد الزبادي
   0.24  B122  5 دقائق حلول تربوية للأطفال — ليات جوشي
   0.21  B130  تربية الأبناء إسلامياً وعلمياً — محمد أحمد زناتي
   0.21  B138  كيف تعلمين طفلك الكلام — فرنسوا إيتيان
   0.18  B124  الرياضيات السريعة للأطفال — بيل هاندلي

💬 answer:
إليك بعض الكتب التي قد تساعدك في تربية أطفالك:

- **«5 دقائق حلول تربوية للأطفال» — ليات جوشي:** يقدم أفكارًا عملية ومختصرة للتعامل مع مواقف التربية اليومية.
- **«تربية الأبناء إسلامياً وعلمياً» — محمد أحمد زناتي:** يناقش احتياجات الأطفال ويؤكد دور العلم والأخلاق والدين في تربيتهم.
- **«صورة الطفولة في التربية الإسلامية» — إبراهيم الخطيب وأحمد الزبادي:** يتناول مكانة الطفل والأسرة وأسس التربية من منظور إسلامي.


❓ هل عندكم رواية هاري بوتر؟

🔎 tool call 1: search_books('رواية هاري بوتر، سلسلة هاري بوتر لج. ك. رولينغ')
   0.12  B198  مدخل إلى أنظمة إلكتر

In [4]:
# Step 4 — a first CODE check (preview of Milestone 4): are the recommended titles real catalog books?
import re
from bookrec.search import normalize_arabic

catalog_titles = [normalize_arabic(t) for t in agent.search.books["title"]]

def recommended_titles(answer):
    """Titles the answer highlights: «quoted» if present, otherwise **bold**."""
    titles = re.findall(r"«(.+?)»", answer) or re.findall(r"\*\*(.+?)\*\*", answer)
    return [t.strip(" :.-–—*«»") for t in titles]

def in_catalog(title):
    t = normalize_arabic(title)
    return any(t == c or t in c or c in t for c in catalog_titles)

for name, r in [("historical novel", r1), ("parenting", r2), ("Harry Potter (trap)", r3), ("Mahfouz (trap)", r4)]:
    titles = recommended_titles(r["answer"])
    invented = [t for t in titles if not in_catalog(t)]
    print(f"{name:22s} recommended {len(titles)} | NOT in catalog: {len(invented)} {invented}")

historical novel       recommended 1 | NOT in catalog: 0 []
parenting              recommended 3 | NOT in catalog: 0 []
Harry Potter (trap)    recommended 0 | NOT in catalog: 0 []
Mahfouz (trap)         recommended 5 | NOT in catalog: 5 ['الثلاثية', 'زقاق المدق', 'اللص والكلاب', 'ميرامار', 'أولاد حارتنا']
